# 🐄 Cattle Anomaly Detection — LSTM Autoencoder Retraining

This notebook retrains the **LSTM Autoencoder** model used for real-time cattle health anomaly detection.

### Pipeline
1. Load labelled cattle sensor dataset
2. Normalize features using saved global scaler
3. Train LSTM Autoencoder **only on normal data**
4. Find optimal anomaly threshold (maximising F1 score)
5. Evaluate and save the retrained model


## 1. Imports & Configuration

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc
)

# ── Config ────────────────────────────────────────
MODEL_DIR        = "dl_service/models"
DATA_PATH        = "dl_service/data/cattle_training_data.csv"
FEATURES         = ["temperature", "humidity", "heartRate",
                    "distance", "hour", "day_of_week"]
SEQUENCE_LENGTH  = 30
N_FEATURES       = 6
EPOCHS           = 50
BATCH_SIZE       = 32
PATIENCE         = 5

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print("Configuration loaded ✅")


## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

# Fix column naming
if "heartRate" not in df.columns and "heart_rate" in df.columns:
    df["heartRate"] = df["heart_rate"]
df["heartRate"] = df["heartRate"].fillna(0)
df = df.dropna(subset=FEATURES).sort_values("timestamp").reset_index(drop=True)

normal_count   = (df["is_anomaly"] == 0).sum()
anomaly_count  = (df["is_anomaly"] == 1).sum()
total          = len(df)

print(f"Total rows   : {total:,}")
print(f"Normal rows  : {normal_count:,}  ({normal_count/total*100:.1f}%)")
print(f"Anomaly rows : {anomaly_count:,}  ({anomaly_count/total*100:.1f}%)")
df.head()


In [ ]:
# Visualise temperature distribution: normal vs anomaly
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Sensor Distribution: Normal vs Anomaly", fontsize=14, fontweight="bold")

for ax, feat, color in zip(axes,
        ["temperature", "heartRate", "humidity"],
        ["steelblue", "tomato", "seagreen"]):
    df[df["is_anomaly"]==0][feat].hist(ax=ax, bins=40, alpha=0.6, label="Normal",  color=color)
    df[df["is_anomaly"]==1][feat].hist(ax=ax, bins=40, alpha=0.6, label="Anomaly", color="red")
    ax.set_title(feat); ax.legend()

plt.tight_layout()
plt.show()


## 3. Preprocessing — Normalise & Create Sequences

In [ ]:
scalers   = joblib.load(os.path.join(MODEL_DIR, "scalers.joblib"))
scaler    = scalers["global"]
X_all     = scaler.transform(df[FEATURES].values)
labels_all = df["is_anomaly"].values
print("Global scaler loaded ✅")

def create_sequences(X, labels=None):
    seqs, lbls = [], []
    for i in range(len(X) - SEQUENCE_LENGTH + 1):
        seqs.append(X[i : i + SEQUENCE_LENGTH])
        if labels is not None:
            lbls.append(labels[i + SEQUENCE_LENGTH - 1])
    return np.array(seqs), (np.array(lbls) if labels is not None else None)

# Train on NORMAL data only — key principle of unsupervised anomaly detection
X_normal  = X_all[labels_all == 0]
X_train, _ = create_sequences(X_normal)
print(f"Training sequences (normal only): {len(X_train):,}")


## 4. LSTM Autoencoder Architecture

In [ ]:
tf.random.set_seed(42)

model = models.Sequential([
    layers.Input(shape=(SEQUENCE_LENGTH, N_FEATURES)),
    # ── Encoder ──────────────────────────────────
    layers.LSTM(64, activation="tanh", return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32, activation="tanh", return_sequences=False),
    # ── Bottleneck ───────────────────────────────
    layers.RepeatVector(SEQUENCE_LENGTH),
    # ── Decoder ──────────────────────────────────
    layers.LSTM(32, activation="tanh", return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(64, activation="tanh", return_sequences=True),
    layers.TimeDistributed(layers.Dense(N_FEATURES)),
], name="LSTM_Autoencoder")

model.compile(optimizer="adam", loss="mse")
model.summary()


## 5. Model Training

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=PATIENCE,
    restore_best_weights=True, verbose=1
)

history = model.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

print(f"
Training stopped at epoch : {len(history.history[chr(108)+chr(111)+chr(115)+chr(115)])}")
print(f"Best validation loss      : {min(history.history[chr(118)+chr(97)+chr(108)+(chr(95))+chr(108)+chr(111)+chr(115)+chr(115)]):.6f}")


In [ ]:
# Plot training vs validation loss
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"],     label="Training Loss",   color="steelblue")
plt.plot(history.history["val_loss"], label="Validation Loss", color="tomato")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.title("LSTM Autoencoder — Training Loss Curve")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.show()


## 6. Threshold Optimisation & Evaluation

In [ ]:
X_eval, labels_eval = create_sequences(X_all, labels_all)
reconstructions = model.predict(X_eval, verbose=1, batch_size=64)
errors = np.mean(np.square(X_eval - reconstructions), axis=(1, 2))

# Find best threshold by maximising F1 score
best_f1, best_threshold = 0, 0
for t in np.percentile(errors, np.arange(50, 99.5, 0.5)):
    preds = (errors > t).astype(int)
    f1 = f1_score(labels_eval, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_threshold = f1, t

predictions = (errors > best_threshold).astype(int)

print(f"Optimal Threshold : {best_threshold:.6f}")
print(f"Accuracy          : {accuracy_score(labels_eval, predictions)*100:.1f}%")
print(f"Precision         : {precision_score(labels_eval, predictions, zero_division=0)*100:.1f}%")
print(f"Recall            : {recall_score(labels_eval, predictions, zero_division=0)*100:.1f}%")
print(f"F1 Score          : {f1_score(labels_eval, predictions, zero_division=0)*100:.1f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reconstruction error distribution
axes[0].hist(errors[labels_eval==0], bins=60, alpha=0.6, label="Normal",  color="steelblue")
axes[0].hist(errors[labels_eval==1], bins=60, alpha=0.6, label="Anomaly", color="tomato")
axes[0].axvline(best_threshold, color="red", linestyle="--", linewidth=2,
                label=f"Threshold = {best_threshold:.4f}")
axes[0].set_xlabel("Reconstruction Error (MSE)")
axes[0].set_ylabel("Count")
axes[0].set_title("Error Distribution — Normal vs Anomaly")
axes[0].legend()

# Confusion matrix
cm = confusion_matrix(labels_eval, predictions)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["Normal","Anomaly"], yticklabels=["Normal","Anomaly"])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
axes[1].set_title("Confusion Matrix")

plt.suptitle("LSTM Autoencoder — Anomaly Detection Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Save Retrained Model & Threshold

In [ ]:
model.save(os.path.join(MODEL_DIR, "global_model.h5"))
joblib.dump(best_threshold, os.path.join(MODEL_DIR, "threshold_global.joblib"))

print("✅ Model saved      → dl_service/models/global_model.h5")
print("✅ Threshold saved  → dl_service/models/threshold_global.joblib")
print()
print("Restart the Flask backend to use the new model:")
print("  python backend_flask/app.py")
